In [16]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [17]:
import pandas as pd
from src.paths import TRASNFORMED_DATA_DIR

df = pd.read_parquet(TRASNFORMED_DATA_DIR / 'tabular_data.parquet')
df

,rides_previous_672_hours,rides_previous_671_hours,rides_previous_670_hours,rides_previous_669_hours,rides_previous_668_hours,rides_previous_667_hours,rides_previous_666_hours,rides_previous_665_hours,rides_previous_664_hours,rides_previous_663_hours,...,rides_previous_7_hours,rides_previous_6_hours,rides_previous_5_hours,rides_previous_4_hours,rides_previous_3_hours,rides_previous_2_hours,rides_previous_1_hours,pickup_hour,pickup_location_id,target_rides_next_hour
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-01-29,1,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-01-30,1,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,2025-01-31,1,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2025-02-01,1,0.0
4,0.0,1.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-02-02,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88289,0.0,3.0,2.0,4.0,3.0,1.0,2.0,1.0,3.0,3.0,...,4.0,2.0,1.0,1.0,2.0,1.0,2.0,2025-12-27,265,4.0
88290,2.0,5.0,2.0,1.0,6.0,0.0,3.0,0.0,3.0,2.0,...,4.0,8.0,0.0,3.0,3.0,1.0,0.0,2025-12-28,265,1.0
88291,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,8.0,0.0,...,8.0,7.0,1.0,6.0,2.0,7.0,2.0,2025-12-29,265,2.0
88292,1.0,1.0,0.0,1.0,2.0,0.0,1.0,3.0,0.0,0.0,...,3.0,3.0,3.0,4.0,0.0,6.0,1.0,2025-12-30,265,4.0


In [18]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df=df,
    cutoff_data=datetime(2025,6,1,0,0,0),
    target_colum_name='target_rides_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')


X_train.shape=(32226, 674)
y_train.shape=(32226,)
X_test.shape=(56068, 674)
y_test.shape=(56068,)


In [19]:
def average_rides_last_4_weeks(X: pd.DataFrame) -> pd.DataFrame:
    """
    Add one column with the average rides from
    - 7 days ago
    - 14 days ago
    - 21 days ago
    - 28 days ago
    """
    X['average_rides_last_4_weeks'] = 0.25 * (
        X[f'rides_previous_{7 * 24}_hours']
        + X[f'rides_previous_{2 * 7 * 24}_hours']
        + X[f'rides_previous_{3 * 7 * 24}_hours']
        + X[f'rides_previous_{4 * 7 * 24}_hours']
    )

    return X

In [20]:
print(X_train.dtypes)

rides_previous_672_hours           float32
rides_previous_671_hours           float32
rides_previous_670_hours           float32
rides_previous_669_hours           float32
rides_previous_668_hours           float32
                                 ...      
rides_previous_3_hours             float32
rides_previous_2_hours             float32
rides_previous_1_hours             float32
pickup_hour                 datetime64[us]
pickup_location_id                   int32
Length: 674, dtype: object


In [21]:
from sklearn.preprocessing import FunctionTransformer

add_feature_average_rides_last_4_weeks = FunctionTransformer(
    func=average_rides_last_4_weeks,
    validate=False
)

In [22]:
add_feature_average_rides_last_4_weeks.fit_transform(X_train)

,rides_previous_672_hours,rides_previous_671_hours,rides_previous_670_hours,rides_previous_669_hours,rides_previous_668_hours,rides_previous_667_hours,rides_previous_666_hours,rides_previous_665_hours,rides_previous_664_hours,rides_previous_663_hours,...,rides_previous_7_hours,rides_previous_6_hours,rides_previous_5_hours,rides_previous_4_hours,rides_previous_3_hours,rides_previous_2_hours,rides_previous_1_hours,pickup_hour,pickup_location_id,average_rides_last_4_weeks
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-01-29,1,0.00
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-01-30,1,0.00
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,2025-01-31,1,0.00
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2025-02-01,1,0.00
4,0.0,1.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-02-02,1,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32221,0.0,3.0,1.0,1.0,0.0,0.0,1.0,2.0,0.0,0.0,...,8.0,2.0,6.0,4.0,2.0,5.0,0.0,2025-05-27,265,1.50
32222,2.0,2.0,0.0,2.0,1.0,1.0,0.0,6.0,0.0,1.0,...,4.0,10.0,1.0,5.0,1.0,0.0,3.0,2025-05-28,265,2.50
32223,5.0,3.0,1.0,2.0,2.0,0.0,2.0,2.0,1.0,1.0,...,0.0,8.0,4.0,4.0,5.0,6.0,12.0,2025-05-29,265,4.25
32224,1.0,9.0,1.0,1.0,2.0,1.0,2.0,0.0,1.0,0.0,...,6.0,13.0,3.0,5.0,4.0,5.0,8.0,2025-05-30,265,6.50


In [23]:
from sklearn.base import BaseEstimator, TransformerMixin

class TemporalFeaturesEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):

        X_ = X.copy()

        # Generate numeric columns form datetime
        X_['hour'] = X_['pickup_hour'].dt.hour
        X_['day_of_week'] = X_['pickup_hour'].dt.dayofweek

        return X_.drop(columns=['pickup_hour'])

In [24]:
add_temporal_features = TemporalFeaturesEngineer()
add_temporal_features.fit_transform(X_train)

,rides_previous_672_hours,rides_previous_671_hours,rides_previous_670_hours,rides_previous_669_hours,rides_previous_668_hours,rides_previous_667_hours,rides_previous_666_hours,rides_previous_665_hours,rides_previous_664_hours,rides_previous_663_hours,...,rides_previous_6_hours,rides_previous_5_hours,rides_previous_4_hours,rides_previous_3_hours,rides_previous_2_hours,rides_previous_1_hours,pickup_location_id,average_rides_last_4_weeks,hour,day_of_week
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,0.00,0,2
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1,0.00,0,3
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1,0.00,0,4
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,0.00,0,5
4,0.0,1.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1,0.00,0,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32221,0.0,3.0,1.0,1.0,0.0,0.0,1.0,2.0,0.0,0.0,...,2.0,6.0,4.0,2.0,5.0,0.0,265,1.50,0,1
32222,2.0,2.0,0.0,2.0,1.0,1.0,0.0,6.0,0.0,1.0,...,10.0,1.0,5.0,1.0,0.0,3.0,265,2.50,0,2
32223,5.0,3.0,1.0,2.0,2.0,0.0,2.0,2.0,1.0,1.0,...,8.0,4.0,4.0,5.0,6.0,12.0,265,4.25,0,3
32224,1.0,9.0,1.0,1.0,2.0,1.0,2.0,0.0,1.0,0.0,...,13.0,3.0,5.0,4.0,5.0,8.0,265,6.50,0,4


In [25]:
import lightgbm as lgm

In [26]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    add_feature_average_rides_last_4_weeks,
    add_temporal_features,
    lgm.LGBMRegressor()
)
pipeline.fit(X_train,y_train)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.116637 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 160172
[LightGBM] [Info] Number of data points in the train set: 32226, number of used features: 675
[LightGBM] [Info] Start training from score 16.010768


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeaturesengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](675,)","['rides_previous_672_hours','rides_previous_671_hours', 'rides_previous_670_hours',...,'pickup_hour','pickup_location_id', 'average_rides_last_4_weeks']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,675
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...0016A073F9760>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [28]:
predictions = pipeline.predict(X_test)

from sklearn.metrics import mean_absolute_error

test_mae = mean_absolute_error(y_test, predictions)

print(f'{test_mae=:.4f}')

test_mae=4.0595
